# 🎮 Steam Data Warehouse — Medallion Architecture
## Cell 1: Bronze | Cell 2: Silver | Cell 3: Gold

In [1]:
import requests
import time
import json
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# 🔑 Configuration
API_KEY = "YOUR_API_KEY_HERE"
TOP_N = 5 # Keep it at 5 for testing, we will raise it to 1000 later

# 🛡️ Network Armor
retry_strategy = Retry(total=3, status_forcelist=[429, 500, 502, 503, 504], backoff_factor=1)
adapter = HTTPAdapter(max_retries=retry_strategy)
http = requests.Session()
http.mount("https://", adapter)

def extract_steam_bronze_data():
    print("🚀 [TASK START] Extracting Bronze Data...")
    
    # --- Step 1: Get AppIDs ---
    charts_url = f"https://api.steampowered.com/ISteamChartsService/GetMostPlayedGames/v1/?key={API_KEY}"
    try:
        top_ranks = http.get(charts_url, timeout=(10, 30)).json()['response']['ranks'][:TOP_N]
    except Exception as e:
        print(f"❌ [FATAL] Charts API Failed: {e}")
        return

    store_lake = []    # Prices, Tags, Names  (JSON Array for Spark)
    reviews_lake = []  # Positive, Negative, AppID  (JSON Array for Spark)

    # --- Step 2 & 3: Get Store Details AND Reviews ---
    for rank in top_ranks:
        appid = rank['appid']
        print(f"📡 Fetching AppID: {appid}...")
        
        store_url = f"https://store.steampowered.com/api/appdetails?appids={appid}&cc=us"
        reviews_url = f"https://store.steampowered.com/appreviews/{appid}?json=1&language=all&purchase_type=all&num_per_page=0"
        
        try:
            store_res = http.get(store_url, timeout=(10, 30)).json()
            time.sleep(1.0)
            
            review_res = http.get(reviews_url, timeout=(10, 30)).json()
            time.sleep(1.0)
            
            if store_res[str(appid)]['success']:
                game_data = store_res[str(appid)]['data']
                game_data['appid'] = appid
                game_data['live_peak_players'] = rank.get('peak_in_game', 0)
                store_lake.append(game_data)
                print(f"✅ [SUCCESS] {game_data['name']} secured. (USD: {game_data.get('price_overview', {}).get('final_formatted', 'Free')})")
                
                if review_res.get('success') == 1:
                    summary = review_res.get('query_summary', {})
                else:
                    summary = {"total_positive": 0, "total_negative": 0}
                summary['appid'] = appid
                reviews_lake.append(summary)
                
        except Exception as e:
            print(f"⚠️ [WARNING] Failed AppID {appid}: {e}")
            
    # --- Step 4: Save TWO separate JSON files ---
    store_path = "data/bronze/store_raw.json"
    reviews_path = "data/bronze/reviews_raw.json"
    
    with open(store_path, "w", encoding="utf-8") as f:
        json.dump(store_lake, f, indent=4, ensure_ascii=False)
    
    with open(reviews_path, "w", encoding="utf-8") as f:
        json.dump(reviews_lake, f, indent=4, ensure_ascii=False)
    
    print(f"🎉 [TASK COMPLETE] Store data saved to {store_path}")
    print(f"🎉 [TASK COMPLETE] Reviews data saved to {reviews_path}")

# Execute
extract_steam_bronze_data()

🚀 [TASK START] Extracting Bronze Data...
📡 Fetching AppID: 730...
✅ [SUCCESS] Counter-Strike 2 secured. (USD: Free)
📡 Fetching AppID: 578080...
✅ [SUCCESS] PUBG: BATTLEGROUNDS secured. (USD: Free)
📡 Fetching AppID: 570...
✅ [SUCCESS] Dota 2 secured. (USD: Free)
📡 Fetching AppID: 2868840...
✅ [SUCCESS] Slay the Spire 2 secured. (USD: $24.99)
📡 Fetching AppID: 431960...
✅ [SUCCESS] Wallpaper Engine secured. (USD: $4.99)
🎉 [TASK COMPLETE] Store data saved to data/bronze/store_raw.json
🎉 [TASK COMPLETE] Reviews data saved to data/bronze/reviews_raw.json


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, BooleanType, FloatType, StringType, StructType
import re
import os, urllib.request

# ============================================================
# 🛡️ Safe Struct Field Extractor
# Some games return requirements as "" instead of {"minimum": ...}
# Spark infers the column as STRING, so .minimum crashes.
# This checks the schema first and returns None if it's not a struct.
# ============================================================
def safe_struct_field(df, col_name, field_name, alias_name, transform_udf=None):
    """Safely extract a nested field from a column that may be STRING or STRUCT."""
    try:
        col_type = df.schema[col_name].dataType
        if isinstance(col_type, StructType) and field_name in col_type.names:
            expr = F.col(f"{col_name}.{field_name}")
            if transform_udf:
                expr = transform_udf(expr)
            return expr.alias(alias_name)
    except Exception:
        pass
    return F.lit(None).cast(StringType()).alias(alias_name)

# ============================================================
# 🪄 HTML Stripper UDF - Strips all HTML tags from descriptions
# ============================================================
def strip_html(text):
    if text is None:
        return None
    clean = re.sub(r'<[^>]+>', ' ', text)       # Remove HTML tags
    clean = re.sub(r'\s+', ' ', clean).strip()   # Collapse whitespace
    return clean

def transform_silver_layer():
    print("🚀 [TASK START] Igniting PySpark Engine...")
    
    # 📦 Auto-download JDBC driver if missing (Needed for Jupiter JVM initialization)
    JDBC_DRIVER_PATH = "drivers/postgresql-42.7.4.jar"
    if not os.path.exists(JDBC_DRIVER_PATH):
        os.makedirs("drivers", exist_ok=True)
        print("📥 Downloading PostgreSQL JDBC driver...")
        urllib.request.urlretrieve(
            "https://jdbc.postgresql.org/download/postgresql-42.7.4.jar",
            JDBC_DRIVER_PATH
        )
    
    abs_jar = os.path.abspath(JDBC_DRIVER_PATH)
    
    # 1. Initialize Spark
    spark = SparkSession.builder \
        .appName("Steam_Medallion_Silver") \
        .config("spark.jars", abs_jar) \
        .config("spark.driver.extraClassPath", abs_jar) \
        .config("spark.executor.extraClassPath", abs_jar) \
        .getOrCreate()

    # Register HTML stripper as a Spark UDF
    strip_html_udf = F.udf(strip_html, StringType())

    # 2. Load the Bronze Data
    df_store = spark.read.option("multiline", "true").json("data/bronze/store_raw.json")
    df_reviews = spark.read.option("multiline", "true").json("data/bronze/reviews_raw.json")

    # Cast appid to Integer on both sides so the join types match
    df_store = df_store.withColumn("appid", F.col("appid").cast(IntegerType()))
    df_reviews = df_reviews.withColumn("appid", F.col("appid").cast(IntegerType()))

    print("🔗 [PROCESS] Joining Store Data with Review Data...")
    df_joined = df_store.join(df_reviews, on="appid", how="left")

    # ============================================================
    # 📊 TABLE 1: games_main.csv — The Master Fact Table (~40 cols)
    # ============================================================
    print("🪄 [PROCESS] Forging games_main.csv...")
    
    df_main = df_joined.select(
        F.col("appid").alias("AppID"),
        F.col("name").alias("Name"),
        F.col("type").alias("Type"),
        F.col("is_free").cast(BooleanType()).alias("Is_Free"),
        F.coalesce(F.col("required_age").cast(IntegerType()), F.lit(0)).alias("Required_Age"),
        
        # 🪄 SPELL: Strip HTML from description fields
        strip_html_udf(F.col("short_description")).alias("Short_Description"),
        strip_html_udf(F.col("detailed_description")).alias("Detailed_Description"),
        strip_html_udf(F.col("about_the_game")).alias("About_The_Game"),
        strip_html_udf(F.col("supported_languages")).alias("Supported_Languages"),
        
        # Press reviews (some games have critic quotes with newlines)
        strip_html_udf(F.col("reviews")).alias("Press_Reviews"),
        
        # 🌐 Website
        F.col("website").alias("Website"),
        
        # 🖼️ ALL the Images
        F.col("header_image").alias("Header_Image"),
        F.col("capsule_image").alias("Capsule_Image"),
        F.col("capsule_imagev5").alias("Capsule_Image_V5"),
        F.col("background").alias("Background_Image"),
        F.col("background_raw").alias("Background_Raw"),
        
        # 💰 Pricing (Full breakdown)
        (F.coalesce(F.col("price_overview.final"), F.lit(0)) / 100).cast(FloatType()).alias("Price_USD"),
        F.col("price_overview.final_formatted").alias("Price_Formatted"),
        F.col("price_overview.currency").alias("Currency"),
        (F.coalesce(F.col("price_overview.initial"), F.lit(0)) / 100).cast(FloatType()).alias("Initial_Price_USD"),
        F.coalesce(F.col("price_overview.discount_percent"), F.lit(0)).alias("Discount_Percent"),
        
        # 🖥️ Platforms
        F.coalesce(F.col("platforms.windows"), F.lit(False)).alias("Is_Windows"),
        F.coalesce(F.col("platforms.mac"), F.lit(False)).alias("Is_Mac"),
        F.coalesce(F.col("platforms.linux"), F.lit(False)).alias("Is_Linux"),
        
        # 👨‍💻 Developers & Publishers (Array → comma-separated)
        F.concat_ws(", ", F.col("developers")).alias("Developers"),
        F.concat_ws(", ", F.col("publishers")).alias("Publishers"),
        
        # 🔧 System Requirements (HTML stripped)
        # Some games return requirements as "" instead of a struct,
        # so safe_struct_field() checks the schema before dot-access.
        safe_struct_field(df_joined, "pc_requirements", "minimum", "PC_Requirements_Min", strip_html_udf),
        safe_struct_field(df_joined, "pc_requirements", "recommended", "PC_Requirements_Rec", strip_html_udf),
        safe_struct_field(df_joined, "mac_requirements", "minimum", "Mac_Requirements_Min", strip_html_udf),
        safe_struct_field(df_joined, "mac_requirements", "recommended", "Mac_Requirements_Rec", strip_html_udf),
        safe_struct_field(df_joined, "linux_requirements", "minimum", "Linux_Requirements_Min", strip_html_udf),
        safe_struct_field(df_joined, "linux_requirements", "recommended", "Linux_Requirements_Rec", strip_html_udf),
        
        # 📅 Release Date
        F.col("release_date.date").alias("Release_Date"),
        F.coalesce(F.col("release_date.coming_soon"), F.lit(False)).alias("Coming_Soon"),
        
        # 📜 Legal (often has <br> and newlines)
        strip_html_udf(F.col("legal_notice")).alias("Legal_Notice"),

        # 🆘 Support Info
        F.col("support_info.url").alias("Support_URL"),
        F.col("support_info.email").alias("Support_Email"),
        
        # ⭐ Metacritic
        F.col("metacritic.score").alias("Metacritic_Score"),
        F.col("metacritic.url").alias("Metacritic_URL"),
        
        # 👍 Recommendations & Achievements totals
        F.coalesce(F.col("recommendations.total"), F.lit(0)).alias("Recommendations_Total"),
        F.coalesce(F.col("achievements.total"), F.lit(0)).alias("Achievements_Total"),
        
        # 🔞 Content Descriptors
        strip_html_udf(F.col("content_descriptors.notes")).alias("Content_Descriptor_Notes"),
        
        # 📝 Reviews from reviews_raw.json (joined data)
        F.coalesce(F.col("total_positive"), F.lit(0)).alias("Positive_Reviews"),
        F.coalesce(F.col("total_negative"), F.lit(0)).alias("Negative_Reviews"),
        F.col("review_score").alias("Review_Score"),
        F.col("review_score_desc").alias("Review_Score_Desc"),
        F.coalesce(F.col("total_reviews"), F.lit(0)).alias("Total_Reviews"),
        
        # 🎮 Live Player Count
        F.coalesce(F.col("live_peak_players"), F.lit(0)).alias("Current_Players")
    )
    
    df_main.coalesce(1).write.csv("data/silver/games_main", header=True, mode="overwrite")
    print("✅ [SAVED] games_main.csv")

    # ============================================================
    # 🎭 TABLE 2: games_genres.csv — Exploded genres
    # ============================================================
    print("🪄 [PROCESS] Forging games_genres.csv...")
    
    df_genres = df_store.select(
        F.col("appid").alias("AppID"),
        F.explode_outer(F.col("genres")).alias("genre")
    ).select(
        "AppID",
        F.col("genre.id").alias("Genre_ID"),
        F.col("genre.description").alias("Genre_Name")
    ).filter(F.col("Genre_ID").isNotNull())
    
    df_genres.coalesce(1).write.csv("data/silver/games_genres", header=True, mode="overwrite")
    print("✅ [SAVED] games_genres.csv")

    # ============================================================
    # 🏷️ TABLE 3: games_categories.csv — Exploded categories
    # ============================================================
    print("🪄 [PROCESS] Forging games_categories.csv...")
    
    df_categories = df_store.select(
        F.col("appid").alias("AppID"),
        F.explode_outer(F.col("categories")).alias("cat")
    ).select(
        "AppID",
        F.col("cat.id").alias("Category_ID"),
        F.col("cat.description").alias("Category_Name")
    ).filter(F.col("Category_ID").isNotNull())
    
    df_categories.coalesce(1).write.csv("data/silver/games_categories", header=True, mode="overwrite")
    print("✅ [SAVED] games_categories.csv")

    # ============================================================
    # 📸 TABLE 4: games_screenshots.csv — All screenshot URLs
    # ============================================================
    print("🪄 [PROCESS] Forging games_screenshots.csv...")
    
    df_screenshots = df_store.select(
        F.col("appid").alias("AppID"),
        F.explode_outer(F.col("screenshots")).alias("ss")
    ).select(
        "AppID",
        F.col("ss.id").alias("Screenshot_ID"),
        F.col("ss.path_thumbnail").alias("Thumbnail_URL"),
        F.col("ss.path_full").alias("Full_URL")
    ).filter(F.col("Screenshot_ID").isNotNull())
    
    df_screenshots.coalesce(1).write.csv("data/silver/games_screenshots", header=True, mode="overwrite")
    print("✅ [SAVED] games_screenshots.csv")

    # ============================================================
    # 🎬 TABLE 5: games_movies.csv — All trailer/video data
    # ============================================================
    print("🪄 [PROCESS] Forging games_movies.csv...")
    
    df_movies = df_store.select(
        F.col("appid").alias("AppID"),
        F.explode_outer(F.col("movies")).alias("mov")
    ).select(
        "AppID",
        F.col("mov.id").alias("Movie_ID"),
        F.col("mov.name").alias("Movie_Name"),
        F.col("mov.thumbnail").alias("Thumbnail"),
        F.col("mov.highlight").alias("Highlight")
    ).filter(F.col("Movie_ID").isNotNull())
    
    df_movies.coalesce(1).write.csv("data/silver/games_movies", header=True, mode="overwrite")
    print("✅ [SAVED] games_movies.csv")

    # ============================================================
    # 🧩 TABLE 6: games_dlc.csv — DLC AppIDs
    # ============================================================
    print("🪄 [PROCESS] Forging games_dlc.csv...")
    
    df_dlc = df_store.select(
        F.col("appid").alias("AppID"),
        F.explode_outer(F.col("dlc")).alias("DLC_AppID")
    ).filter(F.col("DLC_AppID").isNotNull())
    
    df_dlc.coalesce(1).write.csv("data/silver/games_dlc", header=True, mode="overwrite")
    print("✅ [SAVED] games_dlc.csv")

    # ============================================================
    # 🏆 TABLE 7: games_achievements.csv — Highlighted achievements
    # ============================================================
    print("🪄 [PROCESS] Forging games_achievements.csv...")
    
    df_achievements = df_store.select(
        F.col("appid").alias("AppID"),
        F.explode_outer(F.col("achievements.highlighted")).alias("ach")
    ).select(
        "AppID",
        F.col("ach.name").alias("Achievement_Name"),
        F.col("ach.path").alias("Achievement_Icon")
    ).filter(F.col("Achievement_Name").isNotNull())
    
    df_achievements.coalesce(1).write.csv("data/silver/games_achievements", header=True, mode="overwrite")
    print("✅ [SAVED] games_achievements.csv")

    # ============================================================
    print(f"🎉 [TASK COMPLETE] All 7 Silver tables forged in data/silver/")
    spark.stop()

if __name__ == "__main__":
    transform_silver_layer()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as F
import os, urllib.request, sys

# 🔑 Database Configuration (reads from .env / environment variables)
DB_URL  = os.getenv("DB_URL",  "jdbc:postgresql://postgres_general:5432/sessiondb")
DB_USER = os.getenv("DB_USER", "admin")
DB_PASS = os.getenv("DB_PASS", "admin")

# 📦 Auto-download JDBC driver if missing
JDBC_DRIVER_PATH = "drivers/postgresql-42.7.4.jar"
if not os.path.exists(JDBC_DRIVER_PATH):
    os.makedirs("drivers", exist_ok=True)
    print("📥 Downloading PostgreSQL JDBC driver...")
    urllib.request.urlretrieve(
        "https://jdbc.postgresql.org/download/postgresql-42.7.4.jar",
        JDBC_DRIVER_PATH
    )
    print("✅ JDBC driver downloaded.")

# ============================================================
# 📋 Silver → Gold Table Mapping
# Each entry defines:
#   - silver_path: where the CSV lives
#   - gold_table:  target PostgreSQL table name
#   - key_cols:    composite key for upsert deduplication
#
# 🐛 FIX: Previously ALL tables used only ["AppID"] as the key,
#    which collapsed child tables (genres, categories, achievements,
#    screenshots, movies, dlc) to 1 row per game.
#    Now each table uses its proper composite key.
# ============================================================
SILVER_TABLES = [
    {
        "silver_path": "data/silver/games_main",
        "gold_table":  "gold_games_main",
        "key_cols":    ["AppID"],
    },
    {
        "silver_path": "data/silver/games_genres",
        "gold_table":  "gold_games_genres",
        "key_cols":    ["AppID", "Genre_ID"],
    },
    {
        "silver_path": "data/silver/games_categories",
        "gold_table":  "gold_games_categories",
        "key_cols":    ["AppID", "Category_ID"],
    },
    {
        "silver_path": "data/silver/games_screenshots",
        "gold_table":  "gold_games_screenshots",
        "key_cols":    ["AppID", "Screenshot_ID"],
    },
    {
        "silver_path": "data/silver/games_movies",
        "gold_table":  "gold_games_movies",
        "key_cols":    ["AppID", "Movie_ID"],
    },
    {
        "silver_path": "data/silver/games_dlc",
        "gold_table":  "gold_games_dlc",
        "key_cols":    ["AppID", "DLC_AppID"],
    },
    {
        "silver_path": "data/silver/games_achievements",
        "gold_table":  "gold_games_achievements",
        "key_cols":    ["AppID", "Achievement_Name"],
    },
]

def upsert_table(spark, silver_path, table_name, key_cols):
    """Load silver CSV, UPSERT into PostgreSQL using composite key deduplication."""
    
    print(f"📥 [LOADING] {silver_path} → {table_name}...")
    
    # 1. Load Today's Silver Data
    df_new = spark.read \
        .option("multiLine", "true") \
        .option("escape", '"') \
        .csv(silver_path, header=True, inferSchema=True)
    
    new_count = df_new.count()
    if new_count == 0:
        print(f"   ⚠️ [SKIP] {silver_path} is empty. Nothing to load.")
        return {"table": table_name, "silver": 0, "gold": 0, "status": "SKIPPED"}
    
    df_new = df_new.withColumn("Last_Updated", F.current_timestamp())
    print(f"   📊 Silver rows loaded: {new_count}")
    
    # 2. Validate that key columns exist in the data
    missing_keys = [k for k in key_cols if k not in df_new.columns]
    if missing_keys:
        msg = f"   ❌ [ERROR] Key columns {missing_keys} not found in {silver_path}. Columns: {df_new.columns}"
        print(msg)
        return {"table": table_name, "silver": new_count, "gold": 0, "status": "KEY_ERROR", "error": msg}
    
    # 3. Try to Load Existing Gold Data from PostgreSQL
    try:
        df_existing = spark.read \
            .format("jdbc") \
            .option("url", DB_URL) \
            .option("dbtable", table_name) \
            .option("user", DB_USER) \
            .option("password", DB_PASS) \
            .option("driver", "org.postgresql.Driver") \
            .load()
        existing_count = df_existing.count()
        print(f"   🔗 Existing table found ({existing_count} rows). Merging...")
    except Exception:
        print(f"   ⚠️ Table '{table_name}' not found. First load.")
        df_existing = spark.createDataFrame([], df_new.schema)
    
    # Force the new CSV data to strictly match the existing DB schema
    for col_name, col_type in df_existing.dtypes:
        if col_name in df_new.columns:
            df_new = df_new.withColumn(col_name, F.col(col_name).cast(col_type))
    
    # 4. UPSERT: Union → Deduplicate by COMPOSITE KEY (keep newest)
    df_combined = df_existing.unionByName(df_new, allowMissingColumns=True)
    
    window_spec = Window.partitionBy(*key_cols).orderBy(F.col("Last_Updated").desc())
    df_final = df_combined.withColumn("row_num", F.row_number().over(window_spec)) \
                          .filter(F.col("row_num") == 1) \
                          .drop("row_num")
    
    gold_count = df_final.count()
    print(f"   📊 Final gold rows (after dedup): {gold_count}")
    
    # 5. Push to PostgreSQL
    df_final.write \
        .format("jdbc") \
        .option("url", DB_URL) \
        .option("dbtable", table_name) \
        .option("user", DB_USER) \
        .option("password", DB_PASS) \
        .option("driver", "org.postgresql.Driver") \
        .mode("overwrite") \
        .save()
    
    print(f"   ✅ {table_name} secured in PostgreSQL ({gold_count} rows).")
    return {"table": table_name, "silver": new_count, "gold": gold_count, "status": "OK"}

def validate_results(results):
    """Post-load validation: check that no data was lost between Silver and Gold."""
    print("\n" + "=" * 60)
    print("🧪 [VALIDATION] Post-Load Integrity Check")
    print("=" * 60)
    
    all_passed = True
    for r in results:
        table  = r["table"]
        silver = r["silver"]
        gold   = r["gold"]
        status = r["status"]
        
        if status == "SKIPPED":
            print(f"   ⏭️  {table}: SKIPPED (empty silver)")
            continue
        elif status == "KEY_ERROR":
            print(f"   ❌ {table}: FAILED — {r.get('error', 'missing key columns')}")
            all_passed = False
            continue
        elif status != "OK":
            print(f"   ❌ {table}: FAILED — {status}")
            all_passed = False
            continue
        
        # Gold should have >= Silver rows (gold accumulates history)
        if gold >= silver:
            print(f"   ✅ {table}: Silver={silver}, Gold={gold} — OK")
        else:
            print(f"   ❌ {table}: Silver={silver}, Gold={gold} — DATA LOSS DETECTED!")
            all_passed = False
    
    print("=" * 60)
    if all_passed:
        print("🎉 [VALIDATION PASSED] All tables match. No data loss.")
    else:
        print("🚨 [VALIDATION FAILED] Some tables have data integrity issues!")
    print("=" * 60 + "\n")
    
    return all_passed

def load_steam_gold():
    print("🚀 [SPARK] Initiating Gold Layer Transfer...")
    
    # Convert to absolute path so Spark can always find it
    abs_jar = os.path.abspath(JDBC_DRIVER_PATH)
    
    spark = SparkSession.builder \
        .appName("Steam_Gold_Loader") \
        .config("spark.jars", abs_jar) \
        .config("spark.driver.extraClassPath", abs_jar) \
        .config("spark.executor.extraClassPath", abs_jar) \
        .getOrCreate()

    results = []
    for entry in SILVER_TABLES:
        try:
            result = upsert_table(
                spark,
                entry["silver_path"],
                entry["gold_table"],
                entry["key_cols"],
            )
            results.append(result)
        except Exception as e:
            print(f"   ❌ [FAILED] {entry['gold_table']}: {e}")
            results.append({
                "table": entry["gold_table"],
                "silver": -1,
                "gold": -1,
                "status": f"EXCEPTION: {e}",
            })
    
    # 🧪 Run post-load validation
    passed = validate_results(results)
    
    print("🎉 [TASK COMPLETE] All Gold tables secured. Power BI / pgAdmin ready.")
    spark.stop()
    
    if not passed:
        sys.exit(1)

if __name__ == "__main__":
    load_steam_gold()

🚀 [SPARK] Initiating Gold Layer Transfer...
📥 [LOADING] data/silver/games_main → gold_games_main...
   📊 Silver rows loaded: 499
   🔗 Existing table found (499 rows). Merging...
   📊 Final gold rows (after dedup): 499
   ✅ gold_games_main secured in PostgreSQL (499 rows).
📥 [LOADING] data/silver/games_genres → gold_games_genres...
   📊 Silver rows loaded: 866
   🔗 Existing table found (495 rows). Merging...
   📊 Final gold rows (after dedup): 866
   ✅ gold_games_genres secured in PostgreSQL (866 rows).
📥 [LOADING] data/silver/games_categories → gold_games_categories...
   📊 Silver rows loaded: 2941
   🔗 Existing table found (497 rows). Merging...
   📊 Final gold rows (after dedup): 2941
   ✅ gold_games_categories secured in PostgreSQL (2941 rows).
📥 [LOADING] data/silver/games_screenshots → gold_games_screenshots...
   📊 Silver rows loaded: 5789
   🔗 Existing table found (498 rows). Merging...
   📊 Final gold rows (after dedup): 5789
   ✅ gold_games_screenshots secured in PostgreSQL (5